In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

In [ ]:
calibration_data_path = os.path.normpath('~/.ros/calibration.txt')
# read calibration data, first raw is the header
calibration_data = pd.read_csv(calibration_data_path, sep=' ')
calibration_data.dropna(axis=1, inplace=True)
calibration_data
## drop rows if at least one value is bigger then 0.3
calibration_data = calibration_data[(calibration_data < 0.3).all(axis=1)]
calibration_data

In [ ]:
droll_imu = calibration_data['rollIMU'].values
dpitch_imu = calibration_data['pitchIMU'].values
dyaw_imu = calibration_data['yawIMU'].values
droll_icp = calibration_data['rollICP'].values
dpitch_icp = calibration_data['pitchICP'].values
dyaw_icp = calibration_data['yawICP'].values

In [ ]:
# plot the corresponding increments
plt.figure(figsize=(20, 10))
plt.plot(droll_imu, label='rollIMU')
plt.plot(droll_icp, label='rollICP')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(dyaw_imu, label='yawIMU')
plt.plot(dyaw_icp, label='yawICP')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(dpitch_imu, label='pitchIMU')
plt.plot(dpitch_icp, label='pitchICP')
plt.legend()
plt.show()

In [ ]:
# get rotation axis fopr IMU and ICP
def rpy_to_quat(roll, pitch, yaw):
    cy = np.cos(yaw * 0.5)
    sy = np.sin(yaw * 0.5)
    cp = np.cos(pitch * 0.5)
    sp = np.sin(pitch * 0.5)
    cr = np.cos(roll * 0.5)
    sr = np.sin(roll * 0.5)

    q = np.zeros(4)
    q[0] = cy * cp * cr + sy * sp * sr
    q[1] = cy * cp * sr - sy * sp * cr
    q[2] = sy * cp * sr + cy * sp * cr
    q[3] = sy * cp * cr - cy * sp * sr
    return q


dq_imu = np.array([rpy_to_quat(roll, pitch, yaw) for roll, pitch, yaw in zip(droll_imu, dpitch_imu, dyaw_imu)])
dq_icp = np.array([rpy_to_quat(roll, pitch, yaw) for roll, pitch, yaw in zip(droll_icp, dpitch_icp, dyaw_icp)])
axis_imu = [q[1:]/np.linalg.norm(q[1:]) for q in dq_imu]
axis_icp = [q[1:]/np.linalg.norm(q[1:]) for q in dq_icp]
# if angle between two vectors is greater than 90 degrees, flip the vector
for i in range(1, len(axis_imu)):
    if np.dot(axis_imu[i], axis_icp[i]) < 0:
        axis_imu[i] = -axis_imu[i]    

rot_angle_imu = [2*np.arccos(q[0]) for q in dq_imu]
rot_angle_icp = [2*np.arccos(q[0]) for q in dq_icp]
# plot the rotation angles
plt.figure(figsize=(20, 10))
plt.plot(rot_angle_imu, label='IMU')
plt.plot(rot_angle_icp, label='ICP')
plt.legend()
plt.show()

In [ ]:
# get rotations where the angle is larger than 0.05
rot_angle_imu = np.array(rot_angle_imu)
rot_angle_icp = np.array(rot_angle_icp)
axis_imu = np.array(axis_imu)
axis_icp = np.array(axis_icp)
indicies = np.where(rot_angle_imu > 0.05)
rot_angle_imu = rot_angle_imu[indicies]
rot_angle_icp = rot_angle_icp[indicies]
axis_imu = axis_imu[indicies]
axis_icp = axis_icp[indicies]

# plot
plt.figure(figsize=(20, 10))
plt.plot(rot_angle_imu, label='IMU')
plt.plot(rot_angle_icp, label='ICP')
plt.legend()
plt.show()


In [ ]:
# select the rotations with the percentage difference in first 50 percent
percentage = 0.5
rotation_angle_diffs = np.abs(rot_angle_imu - rot_angle_icp)
indicies = np.argsort(rotation_angle_diffs)
num_selected = int(len(indicies) * percentage)
selected_indicies = indicies[:num_selected]
selected_rot_angle_imu = rot_angle_imu[selected_indicies]
selected_rot_angle_icp = rot_angle_icp[selected_indicies]
selected_axis_imu = axis_imu[selected_indicies]
selected_axis_icp = axis_icp[selected_indicies]

# plot the selected rotations
plt.figure(figsize=(20, 10))
plt.plot(selected_rot_angle_imu, label='IMU')
plt.plot(selected_rot_angle_icp, label='ICP')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import minimize

def fit_rotation_l2(axis_imu_array, axis_icp_array):
    """
    Fits the rotation matrix between two sets of corresponding vectors (IMU and ICP).

    Args:
        axis_imu_array (np.ndarray): Array of IMU vectors, shape (n, 3).
        axis_icp_array (np.ndarray): Array of ICP vectors, shape (n, 3).

    Returns:
        np.ndarray: 3x3 rotation matrix that aligns IMU to ICP.
    """
    # Ensure input arrays are numpy arrays
    axis_imu_array = np.asarray(axis_imu_array)
    axis_icp_array = np.asarray(axis_icp_array)

    # Compute the cross-covariance matrix
    H = axis_icp_array.T @ axis_imu_array

    # Perform Singular Value Decomposition (SVD)
    U, _, Vt = np.linalg.svd(H)

    # Compute the rotation matrix
    R = U @ Vt

    # Ensure a proper rotation (det(R) = 1)
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R

def fit_rotation_l1(axis_imu_array, axis_icp_array):
    """
    Fits the rotation matrix between two sets of corresponding vectors (IMU and ICP) using L1 optimality.

    Args:
        axis_imu_array (np.ndarray): Array of IMU vectors, shape (n, 3).
        axis_icp_array (np.ndarray): Array of ICP vectors, shape (n, 3).

    Returns:
        np.ndarray: 3x3 rotation matrix that aligns IMU to ICP using L1 optimality.
    """
    # Ensure input arrays are numpy arrays
    axis_imu_array = np.asarray(axis_imu_array)
    axis_icp_array = np.asarray(axis_icp_array)

    # Get the initial guess from the L2 fitting function
    R_init = fit_rotation_l2(axis_imu_array, axis_icp_array)

    # Define the L1 residual function
    def l1_residual(flat_R):
        R = flat_R.reshape(3, 3)
        residuals = axis_icp_array - axis_imu_array @ R.T
        return np.sum(np.abs(residuals))

    # Ensure the rotation matrix constraints
    def rotation_constraint(flat_R):
        R = flat_R.reshape(3, 3)
        return np.linalg.norm(R.T @ R - np.eye(3), ord='fro')

    # Flatten the initial guess for optimization
    R_init_flat = R_init.ravel()

    # Optimize to minimize L1 residuals
    result = minimize(
        l1_residual,
        R_init_flat,
        constraints={"type": "eq", "fun": rotation_constraint},
        options={"maxiter": 1000, "disp": True},
    )

    # Reshape the result back to a 3x3 matrix
    R_opt = result.x.reshape(3, 3)

    # Ensure proper rotation matrix (orthogonal and det = 1)
    U, _, Vt = np.linalg.svd(R_opt)
    R_final = U @ Vt
    if np.linalg.det(R_final) < 0:
        U[:, -1] *= -1
        R_final = U @ Vt

    return R_final


In [ ]:
selected_axis_imu

In [ ]:
#print with a fixed dot format floats
np.set_printoptions(formatter={'float': lambda x: "{0:0.3f}".format(x)})
selected_axis_icp

In [ ]:
# get rotation matrix
R = fit_rotation_l1(selected_axis_imu, selected_axis_icp)
print(R)

In [ ]:
# visualize the axis: IMU are the red arrows, ICP are the blue arrows
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# plot the points and a line for each corresponding pair
for axis_imu, axis_icp in zip(selected_axis_imu, selected_axis_icp):
    ax.scatter(axis_imu[0], axis_imu[1], axis_imu[2], color='r')
    ax.scatter(axis_icp[0], axis_icp[1], axis_icp[2], color='b')
    ax.plot([axis_imu[0], axis_icp[0]], [axis_imu[1], axis_icp[1]], [axis_imu[2], axis_icp[2]], color='g')

# Set plot limits
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])

plt.show()

rotated_axis_imu = selected_axis_imu @ R.T
print(R)

# for tose who reluted to be more then 90 degrees, flip again, then fit again
for i in range(1, len(selected_axis_icp)):
    if np.dot(selected_axis_icp[i], rotated_axis_imu[i]) < 0:
        selected_axis_icp[i] *= -1

R = fit_rotation(selected_axis_imu, selected_axis_icp)
print(R) 
rotated_axis_imu = (R @ selected_axis_imu.T ).T

# the same after rotation
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# plot the points and a line for each corresponding pair
for axis_imu, axis_icp in zip(rotated_axis_imu, selected_axis_icp):
    # ax.quiver(0, 0, 0, axis_imu[0], axis_imu[1], axis_imu[2], color='r')
    # ax.quiver(0, 0, 0, axis_icp[0], axis_icp[1], axis_icp[2], color='b')
    # not vectors, but points
    ax.scatter(axis_imu[0], axis_imu[1], axis_imu[2], color='r')
    ax.scatter(axis_icp[0], axis_icp[1], axis_icp[2], color='b')

    ax.plot([axis_imu[0], axis_icp[0]], [axis_imu[1], axis_icp[1]], [axis_imu[2], axis_icp[2]], color='g')

# Set plot limits
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])


In [ ]:
# print average distance between the axes
distances_first = np.linalg.norm(selected_axis_imu - selected_axis_icp, axis=1)
print(np.mean(distances_first))
# distances to the rotated axes

distances = np.linalg.norm(rotated_axis_imu - selected_axis_icp, axis=1)
print(np.mean(distances))

In [ ]:
R_cur = np.array([
     0.9398852,   -0.0323142,  0.3399583,
        0.0310275,  0.9994760,  0.0092216,
        -0.3400781,  0.0018808,  0.9403953
]).reshape(3, 3)


In [ ]:
R_final = R @ R_cur
print(R_final)

In [ ]:
plt.plot(distances_first)

In [ ]:
def rotation_matrix_to_rpy(R):
    """
    Converts a 3x3 rotation matrix to roll, pitch, and yaw angles (in radians).
    
    Args:
        R (np.ndarray): 3x3 rotation matrix.
    
    Returns:
        tuple: (roll, pitch, yaw) in radians.
    """
    # Ensure the input is a numpy array
    R = np.asarray(R)

    # Check for gimbal lock (pitch close to ±90 degrees)
    if np.isclose(R[2, 0], -1.0):
        pitch = np.pi / 2
        roll = np.arctan2(R[0, 1], R[0, 2])
        yaw = 0  # Yaw is indeterminate in this case
    elif np.isclose(R[2, 0], 1.0):
        pitch = -np.pi / 2
        roll = np.arctan2(-R[0, 1], -R[0, 2])
        yaw = 0  # Yaw is indeterminate in this case
    else:
        # Compute pitch
        pitch = -np.arcsin(R[2, 0])

        # Compute roll and yaw
        roll = np.arctan2(R[2, 1] / np.cos(pitch), R[2, 2] / np.cos(pitch))
        yaw = np.arctan2(R[1, 0] / np.cos(pitch), R[0, 0] / np.cos(pitch))

    return roll, pitch, yaw